# 01 Dataset Preparation

This notebook prepares the common dataset inputs used by the later RADAR–LiDAR velocity analysis.

It includes:

- filtering the TruckScenes metadata according to locally available sensor files;
- building key-frame sensor observation data;
- separating RADAR and LiDAR observations;
- building the annotation-derived reference velocity.

The original analysis logic is retained from `condition_velocity.ipynb`.

Reusable outputs are saved to `results/condition/` so that later notebooks can run independently without repeating this preparation stage.

In [8]:
import sys
sys.path.append("../..")

from pathlib import Path

from truckscenes import TruckScenes

from src.config import DATA_ROOT, METADATA_ROOT
from src.sensor_observation import build_sensor_data_df
from src.annotation_velocity import build_vehicle_reference_df
from src.dataset_filtering import prepare_available_sample_data

## 1. Dataset Preparation

### Build Sensor Observation Data

In [9]:
trucksc = TruckScenes(dataroot=DATA_ROOT)

Auto-detected latest version: v1.2-mini
Loading truckscenes tables for version v1.2-mini...
11 attribute,
18 calibrated_sensor,
27 category,
20090 ego_motion_cabin,
20089 ego_motion_chassis,
20116 ego_pose,
1094 instance,
400 sample,
25750 sample_annotation,
43556 sample_data,
10 scene,
18 sensor,
4 visibility,
10 weather_annotation,
Done loading in 0.899 seconds.
Reverse indexing ...
Done reverse indexing in 0.1 seconds.


f:\CITS3200\Boeing-RADAR-Autonomous-\.venv311\Lib\site-packages\truckscenes\truckscenes.py:100: UserWarning: The visualization dependencies are not installed on your system! Run 'pip install "truckscenes-devkit[all]"'.
  warnings.warn('''The visualization dependencies are not installed on your system! '''


### Filter Locally Available Sensor Data

Filter the metadata according to the sensor files that are actually available in the local dataset before building the sensor observation tables.

In [10]:
available_sample_data, summary, extra_files = prepare_available_sample_data()
display(summary)

sensor_data_df = build_sensor_data_df(
    metadata_root=METADATA_ROOT,
    sample_data_df=available_sample_data,
)

keyframe_df = sensor_data_df[sensor_data_df["Is Key Frame"].eq(True)].copy()
radar_observation_df = keyframe_df[keyframe_df["Modality"].eq("RADAR")].reset_index(drop=True)
lidar_observation_df = keyframe_df[keyframe_df["Modality"].eq("LiDAR")].reset_index(drop=True)

Metadata records                    43556
Available local files               43556
Unavailable locally                     0
Extra / unreferenced local files        0
Name: Dataset Filtering Summary, dtype: int64

In [11]:
if extra_files:
    print("First extra files:")
    print(*list(extra_files)[:20], sep="\n")

In [12]:
vehicle_reference_df = build_vehicle_reference_df(
    trucksc.sample,
    trucksc.sample_annotation,
    trucksc.instance,
    trucksc.category,
)

print("Reference intervals:", len(vehicle_reference_df))

Reference intervals: 15743


In [13]:
RESULT_DIR = Path("../../results/condition")

available_sample_data.to_csv(RESULT_DIR / "available_sample_data.csv", index=False)
keyframe_df.to_csv(RESULT_DIR / "keyframe_df.csv", index=False)
radar_observation_df.to_csv(RESULT_DIR / "radar_observation_df.csv", index=False)
lidar_observation_df.to_csv(RESULT_DIR / "lidar_observation_df.csv", index=False)

In [14]:
vehicle_reference_df.to_csv(RESULT_DIR / "vehicle_reference_df.csv", index=False)